# 04 Paper-Ready Outputs And Course Mapping

Generate final tables/figures for the written paper and connect the analysis to regression, state machines, boosted trees, SHAP, validation, and agentic reproducibility.

In [ ]:
# Colab/local setup. Keep helper functions at the top of every notebook.
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # Optional: set after the public GitHub repo exists.
TOP_N = 100

if IN_COLAB:
    %pip -q install -r requirements.txt || %pip -q install pybaseball pandas numpy scikit-learn matplotlib seaborn pyarrow shap statsmodels xgboost tqdm
    if REPO_URL and not Path("/content/pitch-sequencing").exists():
        !git clone {REPO_URL} /content/pitch-sequencing
    BASE_DIR = Path("/content/pitch-sequencing") if Path("/content/pitch-sequencing").exists() else Path.cwd()
else:
    BASE_DIR = Path.cwd()

os.chdir(BASE_DIR)
sys.path.insert(0, str(BASE_DIR / "code"))

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "output"

for path in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, OUTPUT_DIR / "figures"]:
    path.mkdir(parents=True, exist_ok=True)

def run_step(args):
    """Runs a subprocess command and prints its stdout and stderr."""
    print("Running:", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=BASE_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

def frame_diag(df, label):
    """Prints diagnostic information about a DataFrame (rows, cols, unique pitchers/pitch types)."""
    print(f"{label}: rows={len(df):,}, cols={df.shape[1]:,}")
    if "pitcher_name" in df:
        print(f"{label}: pitchers={df['pitcher_name'].nunique():,}")
    if "pitch_type" in df:
        print(f"{label}: pitch_types={df['pitch_type'].nunique():,}")
        print(df["pitch_type"].value_counts().head(12))
    return df.head()

def merge_diag(left, right, keys, label):
    """Performs a left merge and prints diagnostic information about the merge process."""
    print(f"[merge:{label}] before left_rows={len(left):,}, right_rows={len(right):,}, keys={keys}")
    print(f"[merge:{label}] right_duplicate_keys={right.duplicated(keys).sum():,}")
    merged = left.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    print(f"[merge:{label}] after rows={len(merged):,}, unmatched={merged['_merge'].eq('left_only').sum():,}")
    return merged.drop(columns=["_merge"])

def show_csv(path, rows=8):
    """Reads a CSV file into a pandas DataFrame and displays the first few rows."""
    import pandas as pd
    path = BASE_DIR / path
    print(path)
    df = pd.read_csv(path)
    display(df.head(rows))
    return df

In [ ]:
run_step([sys.executable, "code/paper_analysis.py", "--top-n", str(TOP_N)])

summary = BASE_DIR / "output/paper_ready/paper_ready_summary.json"
print(summary.read_text())
show_csv("output/paper_ready/tables/paper_model_comparison_top_models.csv")
show_csv("output/paper_ready/tables/ols_pooled_fixed_effects_coefficients.csv")
show_csv("output/paper_ready/tables/shap_pitch_type_feature_importance.csv")

Final deliverables live in `output/paper_ready/`.
Use these for the paper's results section and appendix tables.